In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import ConnectionConfig as cc

cc.setupEnvironment()


In [2]:
spark = cc.startLocalCluster("DIM_VEHICLETYPE", 4)
spark.getActiveSession()

In [3]:

df = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "bike_types") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "biketypeid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 1000) \
    .load()

df.write \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "dim_vehicletype") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .mode("overwrite") \
    .save()

df.repartition(1).write.format("parquet").mode("overwrite").saveAsTable("VehicleDim_pq")

df.write.format("delta").mode("overwrite").saveAsTable("dim_vehicletype")

# Verify if the Delta table is saved correctly
df_delta = spark.read.format("delta").table("dim_vehicletype")
df_delta.show()



+----------+-------------------+
|biketypeid|biketypedescription|
+----------+-------------------+
|         1|          Velo Bike|
|         2|        Velo E-Bike|
|         3|               Step|
|         4|            Scooter|
+----------+-------------------+

